# 07 Fine-tune an embedding model (bge-base) on OPP-115
Same recipe as BERT v2 (sigmoid + weighted loss, same policy split, seed 42). Only the **starting model** changes: `BAAI/bge-base-en-v1.5`, a model already trained to make good paragraph embeddings.

Colab: Runtime > Change runtime type > GPU. Run the cells in order. The test cell at the end is run **once**.

In [1]:
!pip -q install transformers datasets accelerate
!git clone -q https://github.com/dralsarrani/PDPLComplianceCopilot repo
%cd repo

/content/repo


In [2]:
import json, re, numpy as np, pandas as pd, torch
from sklearn.metrics import precision_recall_fscore_support
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)

MODEL = "BAAI/bge-base-en-v1.5"
SEED = 42
set_seed(SEED)

df = pd.read_csv("data/opp115_segments.csv")
CATS = list(df.columns[3:])
df["text"] = df["text"].apply(lambda t: re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", str(t))).strip())
split = json.load(open("configs/split.json"))
part = {k: df[df.policy_file.isin(split[k])].reset_index(drop=True) for k in ("train", "val", "test")}
print({k: len(v) for k, v in part.items()})

{'train': 2550, 'val': 579, 'test': 663}


In [3]:
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(d):
    ds = Dataset.from_dict({"text": d.text.tolist(), "labels": d[CATS].values.astype("float32").tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

ds = {k: to_ds(v) for k, v in part.items()}

Y = part["train"][CATS].values
pos_weight = torch.tensor(np.clip((len(Y) - Y.sum(0)) / Y.sum(0), 1, 10), dtype=torch.float)
print(dict(zip(CATS, pos_weight.numpy().round(1))))

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

Map:   0%|          | 0/579 [00:00<?, ? examples/s]

Map:   0%|          | 0/663 [00:00<?, ? examples/s]

{'First Party Collection/Use': np.float32(2.1), 'Third Party Sharing/Collection': np.float32(3.0), 'User Choice/Control': np.float32(9.3), 'User Access, Edit and Deletion': np.float32(10.0), 'Data Retention': np.float32(10.0), 'Data Security': np.float32(10.0), 'Policy Change': np.float32(10.0), 'Do Not Track': np.float32(10.0), 'International and Specific Audiences': np.float32(10.0), 'Other': np.float32(2.7)}


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATS), problem_type="multi_label_classification")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def metrics(p):
    pred = (1 / (1 + np.exp(-p.predictions)) > 0.5).astype(int)
    f = precision_recall_fscore_support(p.label_ids, pred, zero_division=0)[2]
    return {"macro_f1": f.mean()}

args = TrainingArguments(
    output_dir="out_bge", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=2, report_to="none")

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
                          processing_class=tok, compute_metrics=metrics)
trainer.train()

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-base-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.484959,0.671507
2,No log,0.330021,0.758664
3,No log,0.286490,0.809564
4,0.449869,0.277394,0.837902
5,0.449869,0.269970,0.815991
6,0.449869,0.275081,0.833255
7,0.117915,0.287159,0.829151
8,0.117915,0.266780,0.845932
9,0.117915,0.277615,0.849560
10,0.058876,0.300017,0.837110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1920, training_loss=0.17183118959267935, metrics={'train_runtime': 832.5516, 'train_samples_per_second': 36.754, 'train_steps_per_second': 2.306, 'total_flos': 3287914647721032.0, 'train_loss': 0.17183118959267935, 'epoch': 12.0})

In [6]:
def report(split_name):
    out = trainer.predict(ds[split_name])
    probs = 1 / (1 + np.exp(-out.predictions))
    y = out.label_ids.astype(int)
    p, r, f, s = precision_recall_fscore_support(y, (probs > 0.5).astype(int), zero_division=0)
    rep = pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=CATS).round(3)
    rep.loc["macro avg"] = [p.mean(), r.mean(), f.mean(), s.sum()]
    np.save(f"{split_name}_probs_bge.npy", probs)
    return rep.round(3)

report("val")

,precision,recall,f1,support
First Party Collection/Use,0.818,0.915,0.864,177.0
Third Party Sharing/Collection,0.823,0.923,0.870,156.0
User Choice/Control,0.765,0.812,0.788,48.0
"User Access, Edit and Deletion",0.741,0.800,0.769,25.0
Data Retention,0.909,0.714,0.800,14.0
Data Security,0.862,0.926,0.893,27.0
Policy Change,0.810,0.810,0.810,21.0
Do Not Track,1.000,1.000,1.000,6.0
International and Specific Audiences,0.927,0.962,0.944,53.0
Other,0.723,0.796,0.758,157.0


Compare the `macro avg` f1 above with BERT v2 on validation (0.837) and Legal-BERT (0.851 to 0.868). Differences below about 0.02 are noise. Look at the test cell only if you decide to keep this model.

In [7]:
report("test")   # run once

,precision,recall,f1,support
First Party Collection/Use,0.829,0.918,0.872,196.0
Third Party Sharing/Collection,0.728,0.875,0.795,144.0
User Choice/Control,0.585,0.774,0.667,62.0
"User Access, Edit and Deletion",0.808,0.840,0.824,25.0
Data Retention,0.667,0.400,0.500,10.0
Data Security,0.821,0.865,0.842,37.0
Policy Change,0.704,0.826,0.760,23.0
Do Not Track,1.000,0.667,0.800,6.0
International and Specific Audiences,0.831,0.885,0.857,61.0
Other,0.728,0.745,0.737,212.0


# Ensemble check (validation only)

Question: does averaging the predictions of several models beat the best single model?



In [8]:
yv = part["val"][CATS].values
texts = part["val"].text.tolist()
P = {"bge": np.load("val_probs_bge.npy")}

for name, repo_id in [("bert", "dralsarrani/opp115-bert"), ("legal", "dralsarrani/opp115-legalbert")]:
    t = AutoTokenizer.from_pretrained(repo_id)
    m = AutoModelForSequenceClassification.from_pretrained(repo_id).to("cuda").eval()
    out = []
    for i in range(0, len(texts), 32):
        enc = t(texts[i:i+32], truncation=True, max_length=256, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out.append(torch.sigmoid(m(**enc).logits).cpu().numpy())
    P[name] = np.vstack(out)

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/702k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
combos = {"bert": ["bert"], "legal": ["legal"], "bge": ["bge"],
          "bert+legal": ["bert", "legal"], "bert+bge": ["bert", "bge"],
          "all three": ["bert", "legal", "bge"]}
for k, names in combos.items():
    avg = np.mean([P[n] for n in names], axis=0)
    f = precision_recall_fscore_support(yv, (avg > 0.5).astype(int), zero_division=0)[2]
    print(f"{k:12s} val macro-F1 = {f.mean():.3f}")

bert         val macro-F1 = 0.837
legal        val macro-F1 = 0.858
bge          val macro-F1 = 0.850
bert+legal   val macro-F1 = 0.836
bert+bge     val macro-F1 = 0.847
all three    val macro-F1 = 0.858


**Result:** the best combination (all three models, 0.858) equals Legal-BERT alone (0.858). No combination beat the best single model by the 0.02 we need to rule out noise, so we did not run the ensemble on test and we do not use one in `classify()`.

| Model or combination | Val macro-F1 |
|---|---|
| BERT v2 | 0.837 |
| Legal-BERT | 0.858 |
| bge | 0.850 |
| BERT v2 + Legal-BERT | 0.836 |
| BERT v2 + bge | 0.847 |
| All three | 0.858 |

Validation scores are optimistic, because validation was used to pick epochs. The test set is the honest measure.